# Aula 2 — NumPy
### DS-PY-004 · Técnicas de Programação I

Hoje:

1. Por que NumPy
2. Criando arrays
3. Atributos e `dtype`
4. Indexação, fatiamento e **view vs. copy**
5. Máscaras booleanas
6. Estudo de caso: vendas
7. Matrizes, `axis` e broadcasting
8. Reshape e álgebra linear

In [4]:
#!pip install numpy
import numpy as np
import time

np.random.seed(42)
print("NumPy:", np.__version__)

ModuleNotFoundError: No module named 'numpy'

## 1. Por que NumPy

Vocês já sabem Python. Por que precisaríamos de outra estrutura além da lista?

In [2]:
lista = [1, 2, 3, 4, 5]
print(lista * 2)

[1, 2, 3, 4, 5, 1, 2, 3, 4, 5]


Multiplicar uma lista **repete** a lista. Se fossem preços e você quisesse dobrá-los, o Python te deu outra coisa.

In [3]:
print ("python:")
print([x * 2 for x in lista])          # funciona, mas você escreveu o laço
print ("numpy:")
arr = np.array([1, 2, 3, 4, 5])
print(arr * 2)                          # NumPy

python:
[2, 4, 6, 8, 10]
numpy:


NameError: name 'np' is not defined

> Com NumPy você escreve **o que quer**, não **como percorrer**.

In [5]:
n = 5000000
lista_grande = list(range(n))
array_grande = np.arange(n)

t0 = time.time(); _ = [x ** 2 for x in lista_grande]; t_lista = time.time() - t0
t0 = time.time(); _ = array_grande ** 2;               t_numpy = time.time() - t0

print(f"lista Python : {t_lista:.4f}s")
print(f"NumPy        : {t_numpy:.4f}s")
print(f"ganho        : {t_lista / t_numpy:.2f}x")

lista Python : 0.5201s
NumPy        : 0.0814s
ganho        : 6.39x


### Por que é tão mais rápido?

```
LISTA PYTHON                          ARRAY NUMPY
[ptr]→ objeto int (28 bytes)          [8][8][8][8][8]  ← bytes contíguos
[ptr]→ objeto int
[ptr]→ objeto float                   tudo do MESMO tipo
[ptr]→ objeto str                     tudo GRUDADO na memória
   ↑                                        ↑
espalhado pela memória,               um bloco só, que o processador
cada item é um objeto completo        varre de uma vez
```

1. **Homogêneo** — todos do mesmo tipo, sem checagem item a item
2. **Contíguo** — o processador lê blocos inteiros de uma vez
3. **O laço é em C** — código compilado, não interpretado

É por isso que Pandas, Scikit-learn e quase todo o ecossistema de dados são construídos **em cima** do NumPy.

## 2. Criando arrays

In [10]:
print("de uma lista :", np.array([1, 2, 3]))
print("intervalo    :", np.arange(0, 10, 2))
print("n pontos     :", np.linspace(0, 1, 5))
print("zeros        :", np.zeros(4))
print("uns          :", np.ones(3, dtype=int))
print("constante    :", np.full(3, 7.5))
print("aleatório    :", np.random.rand(3).round(2))

de uma lista : [1 2 3]
intervalo    : [0 2 4 6 8]
n pontos     : [0.   0.25 0.5  0.75 1.  ]
zeros        : [0. 0. 0. 0.]
uns          : [1 1 1]
constante    : [7.5 7.5 7.5]
aleatório    : [0.71 0.02 0.97]


In [6]:
np.array([1,2,3])

array([1, 2, 3])

In [11]:
print(np.arange(0, 1, 0.25))     # arange EXCLUI o fim
print(np.linspace(0, 1, 5))      # linspace INCLUI o fim

[0.   0.25 0.5  0.75]
[0.   0.25 0.5  0.75 1.  ]


## 3. Atributos e `dtype`

In [12]:
a = np.array([[1.5, 2.0, 3.5],
              [4.0, 5.5, 6.0]])

print("shape :", a.shape)
print("ndim  :", a.ndim)
print("size  :", a.size)
print("dtype :", a.dtype)
print("nbytes:", a.nbytes)

shape : (2, 3)
ndim  : 2
size  : 6
dtype : float64
nbytes: 48


> Quando o NumPy reclamar, a primeira coisa a fazer é imprimir o `.shape`. Vale para Pandas e para o resto da carreira.

In [ ]:
inteiros = np.array([1, 2, 3])
print(inteiros)
inteiros[0] = 9.7
print(inteiros)
array_teste = np.array([1,2,3,"banana"])
print(array_teste)
array_teste[0]*2

[1 2 3]
[9 2 3]
['1' '2' '3' 'banana']


'11'

O array é uma caixa de inteiros — só cabe inteiro (pq na criação só tinha inteiro, o tipo é inferido de acordo com os valores de criação, sempre para o tipo mais abrangente, bool -> int -> float -> complex -> string). É o preço da homogeneidade, e origem de bugs silenciosos.

In [17]:
print(np.array([1.5, 2.5, 3.9]).astype(int))    # trunca
print(np.round([1.5, 2.5, 3.9]).astype(int))    # arredonda

[1 2 3]
[2 2 4]


## 4. Indexação, fatiamento e view vs. copy

In [ ]:
v = np.array([10, 20, 30, 40, 50, 60])

print(v[0], v[-1])
print(v[1:4])
print(v[::2])
print(v[::-1])

10 60
[20 30 40]
[20 40 60]
[60 50 40 30 20 10]


In [21]:
original = np.array([1, 2, 3, 4, 5])
print (original)
fatia = original[:3]
print(fatia)
fatia[0] = 999

print("fatia   :", fatia)
print("original:", original)

[1 2 3 4 5]
[1 2 3]
fatia   : [999   2   3]
original: [999   2   3   4   5]


> ⚠️ Fatiar um array NumPy cria uma **view**: uma janela para a mesma memória. Não é cópia.

```
original: [999][ 2 ][ 3 ][ 4 ][ 5 ]
           └──── fatia ────┘
```

É de propósito: copiar milhões de elementos a cada fatiamento acabaria com a performance.

In [22]:
original = np.array([1, 2, 3, 4, 5])
copia = original[:3].copy()
copia[0] = 999

print("cópia   :", copia)
print("original:", original)

cópia   : [999   2   3]
original: [1 2 3 4 5]


**Regra:** se você vai modificar uma fatia e quer preservar o original, use `.copy()`.

## 5. Máscaras booleanas

In [25]:
vendas = np.array([120, 85, 340, 90, 76, 410, 380, 95])
print(vendas)
print(vendas > 300)
print(vendas[vendas > 300])



[120  85 340  90  76 410 380  95]
[False False  True False False  True  True False]
[340 410 380]


In [31]:
print(vendas[(vendas > 100) & (vendas < 400)])
print(vendas[(vendas < 100) | (vendas > 400)])
print(vendas[~(vendas > 100)])

[120 340 380]
[ 85  90  76 410  95]
[85 90 76 95]


> ⚠️ Use `&` `|` `~` — **não** `and` `or` `not`. E parênteses em cada condição.

In [32]:
try:
    vendas[vendas > 100 and vendas < 400]
except ValueError as e:
    print("ValueError:", e)
#and é usado para comparar dois valores e nao dois vetores

ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()


In [34]:
print(vendas>300)
print("quantos acima de 300 :", (vendas > 300).sum())
print("percentual           :", f"{(vendas > 300).mean():.1%}")
print("existe algum > 400?  :", (vendas > 400).any())
print("todos positivos?     :", (vendas > 0).all())

[False False  True False False  True  True False]
quantos acima de 300 : 3
percentual           : 37.5%
existe algum > 400?  : True
todos positivos?     : True


In [ ]:
print(vendas.sum())

np.int64(1596)

Máscara também conta e mede, porque `True` vale 1.

## 6. Estudo de caso: 14 dias de vendas

In [39]:
vendas = np.array([120, 85, 340, 90, 76, 410, 380,
                   95, 110, 305, 60, 88, 420, 395])

print(f"total   : {vendas.sum()}")
print(f"média   : {vendas.mean():.1f}")
print(f"mediana : {np.median(vendas)}")
print(f"desvio  : {vendas.std():.1f}")
print(f"mín/máx : {vendas.min()} / {vendas.max()}")

print(f"Soma vendas > 300   : {vendas[vendas > 300].sum()}")

total   : 2974
média   : 212.4
mediana : 115.0
desvio  : 143.9
mín/máx : 60 / 420
Soma vendas > 300   : 2250


Média 210, mediana 115: há dias muito acima puxando a média. **A média sozinha mente.**

In [41]:
# em QUAIS dias os picos aconteceram
print("valores:", vendas[vendas > 300])
print("dias   :", np.where(vendas > 300)[0] + 1)
print (np.where(vendas > 300)[0])

valores: [340 410 380 305 420 395]
dias   : [ 3  6  7 10 13 14]
[ 2  5  6  9 12 13]


In [45]:
# o array começa numa segunda — existe padrão semanal?
print(vendas)
print(vendas.shape)
semanas = vendas.reshape(2, 7)
print(semanas)
print("\nmédia por dia da semana:", semanas.mean(axis=0).round(1))
print("\nmédia por semana:", semanas.mean(axis=1).round(1))

[120  85 340  90  76 410 380  95 110 305  60  88 420 395]
(14,)
[[120  85 340  90  76 410 380]
 [ 95 110 305  60  88 420 395]]

média por dia da semana: [107.5  97.5 322.5  75.   82.  415.  387.5]

média por semana: [214.4 210.4]


Os picos são sábado e domingo. Um `reshape` transformou números numa descoberta de negócio.

In [46]:
minmax = (vendas - vendas.min()) / (vendas.max() - vendas.min())
z = (vendas - vendas.mean()) / vendas.std()

print("min-max:", minmax.round(2))
print("z-score:", z.round(2))
print("dias acima de 1 desvio:", np.where(z > 1)[0] + 1)

min-max: [0.17 0.07 0.78 0.08 0.04 0.97 0.89 0.1  0.14 0.68 0.   0.08 1.   0.93]
z-score: [-0.64 -0.89  0.89 -0.85 -0.95  1.37  1.16 -0.82 -0.71  0.64 -1.06 -0.86
  1.44  1.27]
dias acima de 1 desvio: [ 6  7 13 14]


| | para quê |
|---|---|
| **min-max** | joga tudo entre 0 e 1 — comparar séries de escalas diferentes |
| **z-score** | quantos desvios da média — achar valores atípicos |

O z-score volta na aula 5.

In [47]:
# np.where(condição, se_verdadeiro, se_falso) — o if/else vetorizado
print(np.where(vendas < 100, 0, vendas))
print(np.where(vendas > 300, 'alto', 'normal'))

[120   0 340   0   0 410 380   0 110 305   0   0 420 395]
['normal' 'normal' 'alto' 'normal' 'normal' 'alto' 'alto' 'normal'
 'normal' 'alto' 'normal' 'normal' 'alto' 'alto']


In [49]:
acum = np.cumsum(vendas)
print(vendas)
print(acum)
print("bateu 1000 no dia:", np.argmax(acum > 1000) + 1)

[120  85 340  90  76 410 380  95 110 305  60  88 420 395]
[ 120  205  545  635  711 1121 1501 1596 1706 2011 2071 2159 2579 2974]
bateu 1000 no dia: 6


> `argmax` num array booleano devolve a posição do **primeiro `True`**. Serve para "quando pela primeira vez...".

## 7. Matrizes, `axis` e broadcasting

In [50]:
notas = np.array([[8.0, 7.5, 9.0, 6.0],
                  [5.5, 6.0, 7.0, 8.5],
                  [9.5, 9.0, 8.0, 7.5],
                  [4.0, 5.0, 6.5, 5.5],
                  [7.0, 8.5, 9.5, 9.0]])
alunos = np.array(['Ana', 'Bruno', 'Carla', 'Diego', 'Elena'])
disciplinas = np.array(['Matemática', 'Estatística', 'Programação', 'Inglês'])

print(notas.shape, "→ 5 alunos, 4 disciplinas")

(5, 4) → 5 alunos, 4 disciplinas


In [51]:
print("Carla em Programação:", notas[2, 2])
print("notas do Bruno      :", notas[1])
print("notas de Inglês     :", notas[:, 3])
print("3 alunos, 2 disciplinas:\n", notas[:3, :2])

Carla em Programação: 8.0
notas do Bruno      : [5.5 6.  7.  8.5]
notas de Inglês     : [6.  8.5 7.5 5.5 9. ]
3 alunos, 2 disciplinas:
 [[8.  7.5]
 [5.5 6. ]
 [9.5 9. ]]


### `axis`

> `axis` não é "em qual direção eu somo". É **qual dimensão eu faço desaparecer**.

```
notas.shape = (5, 4)
               │  │
            axis=0  axis=1
            (linhas) (colunas)

axis=0 → colapsa as LINHAS  → sobra 1 valor por COLUNA → por disciplina (4)
axis=1 → colapsa as COLUNAS → sobra 1 valor por LINHA  → por aluno (5)
```

In [52]:
print("sem axis:", notas.mean().round(2))
print("axis=0  :", notas.mean(axis=0).round(2), "→ por disciplina")
print("axis=1  :", notas.mean(axis=1).round(2), "→ por aluno")

sem axis: 7.32
axis=0  : [6.8 7.2 8.  7.3] → por disciplina
axis=1  : [7.62 6.75 8.5  5.25 8.5 ] → por aluno


O resultado tem o tamanho da dimensão que **sobrou**.

In [55]:
media_aluno = notas.mean(axis=1)
media_disc = notas.mean(axis=0)
print(media_aluno)
print(alunos[2])
print("Melhor aluno        :", alunos[media_aluno.argmax()])
print("Disciplina mais dura:", disciplinas[media_disc.argmin()])
print("Aprovados (>=7)     :", alunos[media_aluno >= 7])
print(f"Taxa de aprovação   : {(media_aluno >= 7).mean():.0%}")
#argmin e argmax devolvel a posição do valor min e max, respectivamente

[7.625 6.75  8.5   5.25  8.5  ]
Carla
Melhor aluno        : Carla
Disciplina mais dura: Matemática
Aprovados (>=7)     : ['Ana' 'Carla' 'Elena']
Taxa de aprovação   : 60%


Repare em `alunos[media_aluno >= 7]`: **a máscara calculada num array filtra outro.**

### Broadcasting

Bônus diferente por disciplina, aplicado a todos os alunos — sem laço.

In [58]:
bonus = np.array([1.0, 0.5, 0.0, 1.5])
print("notas:", notas.shape, "| bonus:", bonus.shape)
print(np.minimum(notas + bonus, 10))

notas: (5, 4) | bonus: (4,)
[[ 9.   8.   9.   7.5]
 [ 6.5  6.5  7.  10. ]
 [10.   9.5  8.   9. ]
 [ 5.   5.5  6.5  7. ]
 [ 8.   9.   9.5 10. ]]


```
notas (5, 4)
bonus    (4)   ← compara da DIREITA para a esquerda: 4 == 4 ✓
              ← a dimensão faltante é esticada
```

In [59]:
try:
    notas + np.array([1, 2, 3, 4, 5])
except ValueError as e:
    print("ValueError:", e)

ValueError: operands could not be broadcast together with shapes (5,4) (5,) 


In [65]:
# bônus POR ALUNO: precisa virar coluna
bonus_aluno = np.array([1.0, 0.5, 0.0, 1.5, 2.0])
print(bonus_aluno)
print(bonus_aluno.reshape(-1, 1))
print("notas")
print(notas)
print("notas+bonus")
print(np.minimum((notas + bonus_aluno.reshape(-1, 1)).round(1),10))

[1.  0.5 0.  1.5 2. ]
[[1. ]
 [0.5]
 [0. ]
 [1.5]
 [2. ]]
notas
[[8.  7.5 9.  6. ]
 [5.5 6.  7.  8.5]
 [9.5 9.  8.  7.5]
 [4.  5.  6.5 5.5]
 [7.  8.5 9.5 9. ]]
notas+bonus
[[ 9.   8.5 10.   7. ]
 [ 6.   6.5  7.5  9. ]
 [ 9.5  9.   8.   7.5]
 [ 5.5  6.5  8.   7. ]
 [ 9.  10.  10.  10. ]]


## 8. Reshape e álgebra linear

In [66]:
base = np.arange(1, 13)
print(base)
print(base.reshape(3, 4), '\n')
print(base.reshape(-1, 2))       # use para nao calcular -1 = "calcule você"

[ 1  2  3  4  5  6  7  8  9 10 11 12]
[[ 1  2  3  4]
 [ 5  6  7  8]
 [ 9 10 11 12]] 

[[ 1  2]
 [ 3  4]
 [ 5  6]
 [ 7  8]
 [ 9 10]
 [11 12]]


In [68]:
print("notas  :", notas.shape)
print(notas)
print("notas.T:", notas.T.shape)
print(notas.T)

notas  : (5, 4)
[[8.  7.5 9.  6. ]
 [5.5 6.  7.  8.5]
 [9.5 9.  8.  7.5]
 [4.  5.  6.5 5.5]
 [7.  8.5 9.5 9. ]]
notas.T: (4, 5)
[[8.  5.5 9.5 4.  7. ]
 [7.5 6.  9.  5.  8.5]
 [9.  7.  8.  6.5 9.5]
 [6.  8.5 7.5 5.5 9. ]]


In [69]:
qtd = np.array([[10, 5, 2],
                [8, 7, 3],
                [12, 4, 1],
                [6, 9, 5]])
precos = np.array([25.0, 40.0, 120.0])

print("faturamento diário:", qtd @ precos)
print("total do período  :", (qtd @ precos).sum())

faturamento diário: [ 690.  840.  580. 1110.]
total do período  : 3220.0


Para cada dia: multiplicou cada quantidade pelo preço do produto e somou. Quatro dias × três produtos em um caractere.

In [71]:
A = np.array([[1, 2], [3, 4]])
B = np.array([[5, 6], [7, 8]])
print(A)
print(B)
print("A * B (elemento a elemento):\n", A * B)
print("A @ B (produto matricial)  :\n", A @ B)

[[1 2]
 [3 4]]
[[5 6]
 [7 8]]
A * B (elemento a elemento):
 [[ 5 12]
 [21 32]]
A @ B (produto matricial)  :
 [[19 22]
 [43 50]]


In [72]:
print("determinante:", np.linalg.det(A).round(2))
print("inversa:\n", np.linalg.inv(A))
print("A @ inv(A):\n", np.round(A @ np.linalg.inv(A), 10))

determinante: -2.0
inversa:
 [[-2.   1. ]
 [ 1.5 -0.5]]
A @ inv(A):
 [[1. 0.]
 [0. 1.]]


### Faltantes (prévia da aula 5)

In [74]:
com_nan = np.array([12.0, np.nan, 15.0, 18.0, np.nan, 22.0])
print(com_nan)
print("mean normal :", com_nan.mean())
print("nanmean     :", np.nanmean(com_nan).round(2))
print(np.isnan(com_nan))
print("onde estão  :", np.where(np.isnan(com_nan))[0])

[12. nan 15. 18. nan 22.]
mean normal : nan
nanmean     : 16.75
[False  True False False  True False]
onde estão  : [1 4]


In [75]:
print(np.nan == np.nan)     # nunca use == para testar NaN
#norma IEEE 754, que define a aritmética de ponto flutuante, determina que NaN é diferente de tudo — inclusive dele mesmo.

False


---
## Resumo

| Ideia | |
|---|---|
| Diga **o que**, não como percorrer | `arr * 2`, `arr > 300` |
| Fatia é **view**, não cópia | `.copy()` quando for modificar |
| Máscara filtra, conta e mede | `arr[arr > x]`, `.sum()`, `.mean()` |
| `axis` = dimensão que **desaparece** | `axis=0` colapsa linhas |
| Broadcasting compara shapes da direita para a esquerda | `notas + bonus` |
| Na dúvida, imprima o `.shape` | sempre |

## Agora é com você

Abra `aula2_numpy.ipynb`. **Sem laços `for`** — é a restrição que força o raciocínio vetorizado.

No fim do notebook está o **mini projeto de NumPy**, entrega em repositório Git.